# SpecML quickstart: load FTIR data, preprocess it, plot it

This notebook is a first hands-on tour of SpecML. The idea is that you **read each cell, run it, then change something and run it again** to see what happens.

What you will do here:
1. Load an FTIR file
2. Look at the raw spectra
3. Apply preprocessing steps one at a time, plotting after each one
4. Look at a tissue image

Cells you are meant to edit are marked with `# EDIT ME`.

## 1. Imports

If `import specml` fails, install the library first by running this in a terminal from the repo root:
```
pip3 install -e .
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from specml.data.spectroscopic_data import SpectroscopicData

# Make plots show up inline in the notebook
%matplotlib inline

## 2. Point at your data file

SpecML can load Agilent FTIR files (`.dmt`, `.dms`, `.dmd`, `.drd`) and Zarr directories.

Replace the path below with a file on your machine.

In [ ]:
FILE_PATH = "/path/to/your/file.dmt"  # EDIT ME

sd = SpectroscopicData(file_path=FILE_PATH)

# `sd.data` is a 2D array: one row per pixel, one column per wavenumber.
# `sd.wavenumbers` is a 1D array of the wavenumber axis (cm^-1).
print("data shape       :", sd.data.shape, "  (n_pixels, n_wavenumbers)")
print("wavenumber range :", f"{sd.wavenumbers.min():.0f} - {sd.wavenumbers.max():.0f} cm^-1")
print("image size       :", f"{sd.ypixels} x {sd.xpixels} pixels")

## 3. A small helper for plotting

We will plot spectra many times below, so we wrap it in a function. Read the body — there is nothing magic happening, just `plt.plot`.

Two conventions worth knowing:
- IR spectra are drawn with **high wavenumbers on the left** (`invert_xaxis`).
- Plotting all pixels would be too many lines, so we sample a few at random.

In [ ]:
def plot_spectra(sd, n=5, title=""):
    """Plot the mean spectrum and a few random individual spectra."""
    rng = np.random.default_rng(0)  # fixed seed so the same pixels are picked
    idx = rng.choice(sd.data.shape[0], size=min(n, sd.data.shape[0]), replace=False)

    fig, ax = plt.subplots(figsize=(9, 4))
    for i in idx:
        ax.plot(sd.wavenumbers, sd.data[i], lw=0.8, alpha=0.6)
    ax.plot(sd.wavenumbers, sd.data.mean(axis=0), color="k", lw=2, label="mean")
    ax.invert_xaxis()
    ax.set_xlabel("Wavenumber (cm$^{-1}$)")
    ax.set_ylabel("Intensity")
    ax.set_title(title)
    ax.legend()
    plt.show()

plot_spectra(sd, title="raw spectra")

## 4. Trim to the fingerprint region

Most biological information sits between ~1000 and 1800 cm$^{-1}$ (the "fingerprint region"). The high-wavenumber end is dominated by C–H and O–H stretches that are noisy and often less informative for tissue classification.

`keep_range(lower, upper)` returns a new `SpectroscopicData` with only that slice. **Try changing the limits and re-running** to see how the plot changes.

In [ ]:
LOWER = 1000  # EDIT ME
UPPER = 1800  # EDIT ME

sd = sd.keep_range(LOWER, UPPER)

print("new shape:", sd.data.shape)
plot_spectra(sd, title=f"after keep_range({LOWER}, {UPPER})")

## 5. Optional: remove CO$_2$ and paraffin

- **CO$_2$** absorbs at ~2350 cm$^{-1}$. If your sample was scanned with imperfect background correction you can see a sharp band there.
- **Paraffin** (wax) used in FFPE sample preparation contributes peaks near 2900, 1470, and 720 cm$^{-1}$.

These are usually only relevant if your kept range still includes those regions. With a 1000–1800 cm$^{-1}$ window the CO$_2$ region is already gone, so `remove_co2()` would do nothing — that is fine.

Set the flags below to `True` to enable them.

In [ ]:
REMOVE_CO2 = False  # EDIT ME
REMOVE_WAX = False  # EDIT ME

if REMOVE_CO2:
    sd = sd.remove_co2()
if REMOVE_WAX:
    sd = sd.remove_wax()

if REMOVE_CO2 or REMOVE_WAX:
    plot_spectra(sd, title="after CO2 / wax removal")
else:
    print("skipped")

## 6. Baseline: shift each spectrum's minimum to zero

The simplest baseline correction: for each pixel, subtract its own minimum. After this every spectrum starts at zero.

It is crude (a real baseline is rarely flat) but it is fast and good enough for many tasks. SpecML also has `baseline_correction` (asymmetric least squares) if you want something fancier.

In [ ]:
sd = sd.adjust_min_to_zero()
plot_spectra(sd, title="after adjust_min_to_zero")

## 7. Vector normalisation

Different pixels can have very different overall intensities (e.g. thicker vs. thinner tissue) even when the *shape* of the spectrum is the same. Vector normalisation divides each spectrum by its L2 norm so they all sit on the unit sphere, leaving only the shape.

After this all spectra should look much more similar in scale.

In [ ]:
sd = sd.normalize(method="vector")  # try "z_score" too and see what changes
plot_spectra(sd, title="after vector normalisation")

## 8. Optional: Savitzky–Golay smoothing

Fits a small polynomial in a sliding window and replaces each point with the polynomial value. Reduces high-frequency noise without flattening peaks the way a moving average would.

- `window_length` must be **odd**. Larger = smoother but blurs sharp peaks.
- `polyorder` < `window_length`. Higher order preserves curvature better.

In [ ]:
DO_SMOOTH = False  # EDIT ME

if DO_SMOOTH:
    sd = sd.smooth_data(window_length=11, polyorder=2, interp=True)
    plot_spectra(sd, title="after Savitzky-Golay smoothing")
else:
    print("skipped")

## 9. Optional: derivative

Derivatives accentuate peak positions and remove slowly-varying baseline contributions:
- **1st derivative**: removes constant offsets, peaks become zero-crossings.
- **2nd derivative**: removes linear baselines, peaks become sharp negative dips.

They also amplify noise — usually you smooth first.

In [ ]:
DO_DERIVATIVE = False  # EDIT ME
DERIV_ORDER = 2        # 1 or 2

if DO_DERIVATIVE:
    sd = sd.apply_derivative(window_length=23, polyorder=5,
                              deriv=DERIV_ORDER, interp=True)
    plot_spectra(sd, title=f"after {DERIV_ORDER}-order derivative")
else:
    print("skipped")

## 10. Look at the tissue as an image

Up to now we have only looked at spectra. We can also map a single number per pixel back into the tissue shape and display it.

`sd.display(lower, upper)` integrates the absorbance between two wavenumbers (the area under the band) and shows the result as a heat-map. The amide I band (1600–1700 cm$^{-1}$) is a good default for biological tissue: it tracks total protein content and outlines the tissue clearly.

In [ ]:
AMIDE_LOWER = 1600  # EDIT ME
AMIDE_UPPER = 1700  # EDIT ME

sd.display(lower=AMIDE_LOWER, upper=AMIDE_UPPER,
           title=f"integrated absorbance {AMIDE_LOWER}-{AMIDE_UPPER} cm$^{{-1}}$",
           color_bar=True)

## Try this next

1. Re-run the notebook with `keep_range(950, 3550)` and `REMOVE_WAX = True`. How do the plots differ?
2. Compare `normalize(method="vector")` to `normalize(method="z_score")`.
3. Turn on smoothing **and** the second derivative. Where do peaks go?
4. Change `AMIDE_LOWER`/`AMIDE_UPPER` to a different band (e.g. 1500-1600 for amide II) and see the image change.
5. Pick a single pixel with `sd.get_spectrum_at(x, y)` and plot just that one.

Once you are comfortable with this, move on to `01_data_loading_and_quality_control.ipynb` for the full workflow.